# RLO Experiments - 8x B200 (Fixed: compile + DataParallel)

**Fixes:**
- torch.compile disabled with DataParallel (incompatible)
- Reduced DataLoader workers
- Memory optimizations

In [ ]:
# =============================================================================
# CELL 1: Setup
# =============================================================================

import os
import sys
import time
import math
import random
import json
import gc
import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['font.size'] = 12

warnings.filterwarnings('ignore')

# GPU optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Paths
IMAGENET_FOLDER = Path("/blue/wdixon/wang.yixuan/lypcdf/imagenet_folder")
RESULTS_DIR = Path("./rlo_results")
RESULTS_DIR.mkdir(exist_ok=True)

# Check GPUs
NUM_GPUS = torch.cuda.device_count()
print(f"Available GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.0f}GB")

# Check data
if (IMAGENET_FOLDER / 'train').exists():
    train_classes = len(list((IMAGENET_FOLDER / 'train').iterdir()))
    print(f"\n✓ ImageFolder ready: {train_classes} classes")
else:
    print(f"\n✗ ImageFolder not found at {IMAGENET_FOLDER}")

In [ ]:
# =============================================================================
# CELL 2: Optimizers
# =============================================================================

class RLO(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.1, 
                 belief_coef=0.1, eps=1e-8):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay, 
                       belief_coef=belief_coef, eps=eps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, wd = group["lr"], group["weight_decay"]
            beta1, beta2 = group["betas"]
            belief, eps = group["belief_coef"], group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["exp_avg"] = torch.zeros_like(p)
                m = state["exp_avg"]
                if wd != 0.0:
                    p.mul_(1.0 - lr * wd)
                c = beta1 * m + (1.0 - beta1) * g
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                d = c.sign() + belief * (delta / delta_norm)
                p.add_(d, alpha=-lr)
                m.mul_(beta2).add_(g, alpha=(1.0 - beta2))
        return loss


class RLO_LambdaA(Optimizer):
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.99, beta3=0.999,
                 weight_decay=0.1, lambda_b=0.1, eps=1e-8, gamma=5.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, beta3=beta3,
                       weight_decay=weight_decay, lambda_b=lambda_b, eps=eps, gamma=gamma)
        super().__init__(params, defaults)
        self._init_sqrt_dim()

    def _init_sqrt_dim(self):
        total = sum(p.numel() for g in self.param_groups for p in g["params"])
        self.sqrt_dim = math.sqrt(total)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        all_smooth_pre, all_belief, all_params = [], [], []
        for group in self.param_groups:
            eps, gamma = group["eps"], group["gamma"]
            beta1, beta2, beta3 = group["beta1"], group["beta2"], group["beta3"]
            lambda_b = group["lambda_b"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)
                    state["s"] = torch.zeros_like(p)
                m, s = state["m"], state["s"]
                s.mul_(beta3).addcmul_(g, g, value=(1.0 - beta3))
                c = beta1 * m + (1.0 - beta1) * g
                smooth = torch.tanh(gamma * c)
                smooth_pre = smooth / (s.sqrt() + eps)
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                belief = lambda_b * (delta / delta_norm)
                all_smooth_pre.append(smooth_pre)
                all_belief.append(belief)
                all_params.append((p, group))
        if not all_params:
            return loss
        s_norm = sum((sp * sp).sum() for sp in all_smooth_pre).sqrt().clamp(min=1e-8)
        scale = self.sqrt_dim / s_norm
        for (p, group), sp, b in zip(all_params, all_smooth_pre, all_belief):
            lr, wd, beta2 = group["lr"], group["weight_decay"], group["beta2"]
            d = scale * sp + b
            state = self.state[p]
            if wd != 0.0:
                p.mul_(1.0 - lr * wd)
            p.add_(d, alpha=-lr)
            state["m"].mul_(beta2).add_(p.grad, alpha=(1.0 - beta2))
        return loss


class SmoothLiftedRLO(Optimizer):
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.99, eta=0.3,
                 weight_decay=0.1, lambda_b=0.1, eps=1e-8, gamma=5.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eta=eta,
                       weight_decay=weight_decay, lambda_b=lambda_b, eps=eps, gamma=gamma)
        super().__init__(params, defaults)
        self._init_sqrt_dim()

    def _init_sqrt_dim(self):
        total = sum(p.numel() for g in self.param_groups for p in g["params"])
        self.sqrt_dim = math.sqrt(total)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        all_s, all_b, all_params = [], [], []
        for group in self.param_groups:
            eps, gamma = group["eps"], group["gamma"]
            beta1, lambda_b = group["beta1"], group["lambda_b"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)
                    state["v"] = torch.zeros_like(p)
                m = state["m"]
                c = beta1 * m + (1.0 - beta1) * g
                s = torch.tanh(gamma * c)
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                b = lambda_b * (delta / delta_norm)
                all_s.append(s)
                all_b.append(b)
                all_params.append((p, group))
        if not all_params:
            return loss
        s_norm = sum((s * s).sum() for s in all_s).sqrt().clamp(min=1e-8)
        scale = self.sqrt_dim / s_norm
        for (p, group), s, b in zip(all_params, all_s, all_b):
            lr, wd, eta, beta2 = group["lr"], group["weight_decay"], group["eta"], group["beta2"]
            d = scale * s + b
            state = self.state[p]
            m, v = state["m"], state["v"]
            if wd != 0.0:
                p.mul_(1.0 - lr * wd)
            v.mul_(1.0 - eta).add_(d, alpha=eta)
            p.add_(v, alpha=-lr)
            m.mul_(beta2).add_(p.grad, alpha=(1.0 - beta2))
        return loss


class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, wd = group['lr'], group['weight_decay']
            beta1, beta2 = group['betas']
            for p in group['params']:
                if p.grad is None:
                    continue
                if wd != 0.0:
                    p.mul_(1.0 - lr * wd)
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                m = state['exp_avg']
                update = (beta1 * m + (1 - beta1) * g).sign()
                p.add_(update, alpha=-lr)
                m.mul_(beta2).add_(g, alpha=1 - beta2)
        return loss


def create_optimizer(model, opt_name, lr, wd):
    params = model.parameters()
    if opt_name == 'adamw':
        return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    elif opt_name == 'lion':
        return Lion(params, lr=lr*0.1, weight_decay=wd*10)
    elif opt_name == 'rlo':
        return RLO(params, lr=lr*0.1, weight_decay=wd*10)
    elif opt_name == 'rlo_lambda_a':
        return RLO_LambdaA(params, lr=lr*0.1, weight_decay=wd*10)
    elif opt_name == 'smooth_lifted_rlo':
        return SmoothLiftedRLO(params, lr=lr*0.1, weight_decay=wd*10)
    raise ValueError(f"Unknown: {opt_name}")

print("Optimizers ready")

In [ ]:
# =============================================================================
# CELL 3: Data Loading
# =============================================================================

def get_imagenet_loaders(
    root: Path,
    batch_size: int = 256,
    num_workers: int = 4,
    use_randaugment: bool = False,
):
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    
    train_transforms = [
        T.RandomResizedCrop(224, interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
    ]
    if use_randaugment:
        train_transforms.append(T.RandAugment(num_ops=2, magnitude=9))
    train_transforms.extend([T.ToTensor(), T.Normalize(MEAN, STD)])
    
    val_transform = T.Compose([
        T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])
    
    train_dataset = ImageFolder(root / 'train', T.Compose(train_transforms))
    val_dataset = ImageFolder(root / 'val', val_transform)
    
    print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
        persistent_workers=False, prefetch_factor=2 if num_workers > 0 else None,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size * 2, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=False, prefetch_factor=2 if num_workers > 0 else None,
    )
    
    return train_loader, val_loader


class GPUMixup:
    def __init__(self, mixup_alpha=0.8, cutmix_alpha=1.0, num_classes=1000):
        self.mixup_alpha, self.cutmix_alpha = mixup_alpha, cutmix_alpha
        self.num_classes = num_classes
        
    @torch.no_grad()
    def __call__(self, x, target):
        use_cutmix = random.random() < 0.5
        alpha = self.cutmix_alpha if use_cutmix else self.mixup_alpha
        lam = np.random.beta(alpha, alpha)
        B = x.size(0)
        index = torch.randperm(B, device=x.device)
        if use_cutmix:
            _, _, H, W = x.shape
            cut_rat = math.sqrt(1.0 - lam)
            cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
            cx, cy = random.randint(0, W), random.randint(0, H)
            x1, y1 = max(cx - cut_w//2, 0), max(cy - cut_h//2, 0)
            x2, y2 = min(cx + cut_w//2, W), min(cy + cut_h//2, H)
            x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
            lam = 1 - ((x2-x1)*(y2-y1) / (W*H))
        else:
            x = lam * x + (1-lam) * x[index]
        y = F.one_hot(target, self.num_classes).float()
        y_perm = F.one_hot(target[index], self.num_classes).float()
        return x, lam * y + (1-lam) * y_perm

print("Data loading ready")

In [ ]:
# =============================================================================
# CELL 4: Models
# =============================================================================

def create_resnet50(num_classes=1000):
    from torchvision.models import resnet50
    return resnet50(weights=None, num_classes=num_classes)


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads, self.head_dim = num_heads, dim // num_heads
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(B, N, C))


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop)
        )
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=1000,
                 embed_dim=768, depth=12, num_heads=12, drop_rate=0.):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)
        self.blocks = nn.ModuleList([Block(embed_dim, num_heads, drop=drop_rate) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat((self.cls_token.expand(B, -1, -1), x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x)[:, 0])


def create_vit_small(num_classes=1000):
    return VisionTransformer(embed_dim=384, depth=12, num_heads=6, num_classes=num_classes)

def create_vit_base(num_classes=1000):
    return VisionTransformer(embed_dim=768, depth=12, num_heads=12, num_classes=num_classes)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ResNet-50: {count_params(create_resnet50())/1e6:.1f}M")
print(f"ViT-S/16: {count_params(create_vit_small())/1e6:.1f}M")
print(f"ViT-B/16: {count_params(create_vit_base())/1e6:.1f}M")

In [ ]:
# =============================================================================
# CELL 5: Training Utilities
# =============================================================================

class CosineScheduler:
    def __init__(self, optimizer, base_lr, total_steps, warmup_steps, min_lr=0):
        self.optimizer, self.base_lr = optimizer, base_lr
        self.total_steps, self.warmup_steps, self.min_lr = total_steps, warmup_steps, min_lr
        self.step_count = 0
        
    def step(self):
        self.step_count += 1
        lr = self._get_lr()
        for g in self.optimizer.param_groups:
            g['lr'] = lr
        return lr
    
    def _get_lr(self):
        if self.step_count < self.warmup_steps:
            return self.base_lr * self.step_count / max(1, self.warmup_steps)
        progress = (self.step_count - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        return self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / max(total, 1), 100.0 * correct / max(total, 1)


def plot_results(results: Dict, title: str, save_path: Path = None):
    colors = {'adamw': '#1f77b4', 'lion': '#ff7f0e', 'rlo': '#2ca02c',
              'rlo_lambda_a': '#d62728', 'smooth_lifted_rlo': '#9467bd'}
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for name, res in results.items():
        h = res.get('history', {})
        c = 'gray'
        for k in colors:
            if k in name:
                c = colors[k]
                break
        
        if 'train_acc' in h:
            axes[0].plot(h['train_acc'], label=name, color=c, linewidth=2)
        if 'val_acc' in h:
            axes[1].plot(h['val_acc'], label=name, color=c, linewidth=2)
        if 'throughput' in h:
            axes[2].plot(h['throughput'], label=name, color=c, linewidth=2)
    
    axes[0].set_title('Training Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].set_title('Validation Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[2].set_title('Throughput (img/s)'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


def print_table(results: Dict, title: str):
    print(f"\n{'='*60}\n{title}\n{'='*60}")
    print(f"{'Optimizer':<25} {'Best':>8} {'Final':>8} {'img/s':>10}")
    print('-'*55)
    for name, res in sorted(results.items(), key=lambda x: x[1].get('best_acc', 0), reverse=True):
        print(f"{name:<25} {res.get('best_acc',0):>7.2f}% {res.get('final_acc',0):>7.2f}% {res.get('avg_throughput',0):>9.0f}")

print("Training utilities ready")

In [ ]:
# =============================================================================
# CELL 6: Main Training Function (FIXED: no compile with DataParallel)
# =============================================================================

def train_model(
    model_name: str,
    opt_name: str,
    epochs: int = 90,
    batch_size: int = 2048,
    lr: float = 1e-3,
    wd: float = 0.05,
    warmup_epochs: int = 5,
    use_randaugment: bool = False,
    use_mixup: bool = False,
    label_smoothing: float = 0.1,
    num_workers: int = 4,
):
    """
    Train with DataParallel.
    
    NOTE: torch.compile is DISABLED because it's incompatible with DataParallel.
    Still fast due to TF32, bfloat16, and large batch sizes.
    """
    set_seed(42)
    gc.collect()
    torch.cuda.empty_cache()
    
    num_gpus = torch.cuda.device_count()
    device = torch.device("cuda:0")
    run_name = f"{model_name}_{opt_name}"
    
    print(f"\n{'='*70}")
    print(f"Training: {run_name}")
    print(f"GPUs: {num_gpus}, Batch: {batch_size}, Workers: {num_workers}")
    print(f"NOTE: torch.compile disabled (incompatible with DataParallel)")
    print(f"{'='*70}")
    
    # Data
    train_loader, val_loader = get_imagenet_loaders(
        IMAGENET_FOLDER, batch_size=batch_size,
        num_workers=num_workers, use_randaugment=use_randaugment,
    )
    
    # Model
    if model_name == "resnet50":
        model = create_resnet50()
    elif model_name == "vit_s16":
        model = create_vit_small()
    elif model_name == "vit_b16":
        model = create_vit_base()
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    model = model.to(device)
    
    # NO torch.compile - incompatible with DataParallel!
    
    if num_gpus > 1:
        model = nn.DataParallel(model)
    
    print(f"Params: {count_params(model)/1e6:.1f}M")
    
    # Optimizer
    optimizer = create_optimizer(model, opt_name, lr, wd)
    actual_lr = optimizer.param_groups[0]['lr']
    print(f"Optimizer: {opt_name}, lr={actual_lr:.2e}")
    
    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader) * warmup_epochs
    scheduler = CosineScheduler(optimizer, actual_lr, total_steps, warmup_steps)
    
    # Loss
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    mixup = GPUMixup() if use_mixup else None
    
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': [], 'throughput': []}
    best_acc = 0.0
    
    print(f"Starting: {len(train_loader)} batches/epoch")
    
    for epoch in range(epochs):
        model.train()
        epoch_start = time.time()
        correct, total, running_loss = 0, 0, 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for step, (images, labels) in enumerate(pbar):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            if mixup:
                images, labels_mixed = mixup(images, labels)
                use_soft = True
            else:
                use_soft = False
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits = model(images)
                if use_soft:
                    loss = -torch.sum(F.log_softmax(logits, 1) * labels_mixed, 1).mean()
                else:
                    loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            pred = logits.argmax(1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
            running_loss += loss.item()
            
            if step % 50 == 0:
                pbar.set_postfix({'loss': f'{running_loss/(step+1):.4f}', 'acc': f'{100*correct/total:.1f}%'})
        
        epoch_time = time.time() - epoch_start
        throughput = len(train_loader) * batch_size / epoch_time
        train_acc = 100 * correct / total
        train_loss = running_loss / len(train_loader)
        
        # Validation
        model_eval = model.module if hasattr(model, 'module') else model
        val_loss, val_acc = evaluate(model_eval, val_loader, criterion, device)
        
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['throughput'].append(throughput)
        
        is_best = val_acc > best_acc
        best_acc = max(val_acc, best_acc)
        
        print(f"Epoch {epoch+1}: Train={train_acc:.1f}%, Val={val_acc:.1f}% {'*' if is_best else ''}, "
              f"Throughput={throughput:.0f} img/s")
        
        if is_best:
            torch.save({'epoch': epoch, 'model_state_dict': model_eval.state_dict(),
                       'best_acc': best_acc}, RESULTS_DIR / f"{run_name}_best.pt")
    
    result = {
        'run_name': run_name, 'model': model_name, 'optimizer': opt_name,
        'best_acc': best_acc, 'final_acc': history['val_acc'][-1],
        'avg_throughput': np.mean(history['throughput']), 'history': history,
    }
    
    with open(RESULTS_DIR / f"{run_name}.json", 'w') as f:
        json.dump(result, f, indent=2, default=str)
    
    del model, optimizer, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    
    return result

print("Training function ready (torch.compile disabled for DataParallel compatibility)")

In [ ]:
# =============================================================================
# CELL 7: Quick Test
# =============================================================================

def quick_test(epochs=3, batch_size=2048):
    print("\n" + "="*70)
    print("QUICK TEST: ResNet-50 + AdamW")
    print("="*70)
    
    result = train_model(
        model_name="resnet50", opt_name="adamw",
        epochs=epochs, batch_size=batch_size,
        lr=0.4, wd=1e-4, warmup_epochs=1,
    )
    
    print(f"\n✓ Quick test complete!")
    print(f"  Best acc: {result['best_acc']:.2f}%")
    print(f"  Throughput: {result['avg_throughput']:.0f} img/s")
    return result

# Run quick test
# quick_result = quick_test()

In [ ]:
# =============================================================================
# CELL 8: ResNet-50 Experiments
# =============================================================================

def run_resnet50_experiments():
    print("\n" + "#"*70)
    print("# ResNet-50 on ImageNet (90 epochs)")
    print("#"*70)
    
    results = {}
    
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        try:
            result = train_model(
                model_name="resnet50", opt_name=opt,
                epochs=90, batch_size=4096,
                lr=0.4, wd=1e-4, warmup_epochs=5,
                use_randaugment=False, use_mixup=False, label_smoothing=0.0,
            )
            results[f"resnet50_{opt}"] = result
            
            # Plot progress
            print_table(results, "ResNet-50 Progress")
            plot_results(results, "ResNet-50 Progress", RESULTS_DIR / "resnet50_progress.png")
            
        except Exception as e:
            print(f"Failed {opt}: {e}")
            import traceback
            traceback.print_exc()
    
    # Final
    print_table(results, "ResNet-50 Final")
    plot_results(results, "ResNet-50 on ImageNet", RESULTS_DIR / "resnet50_final.png")
    
    with open(RESULTS_DIR / "resnet50_all.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    return results

# Run
# resnet_results = run_resnet50_experiments()

In [ ]:
# =============================================================================
# CELL 9: ViT-S/16 Experiments
# =============================================================================

def run_vit_s16_experiments():
    print("\n" + "#"*70)
    print("# ViT-S/16 on ImageNet (300 epochs)")
    print("#"*70)
    
    results = {}
    
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        try:
            result = train_model(
                model_name="vit_s16", opt_name=opt,
                epochs=300, batch_size=2048,
                lr=1e-3, wd=0.05, warmup_epochs=30,
                use_randaugment=True, use_mixup=True, label_smoothing=0.1,
            )
            results[f"vit_s16_{opt}"] = result
            
            print_table(results, "ViT-S/16 Progress")
            plot_results(results, "ViT-S/16 Progress", RESULTS_DIR / "vit_s16_progress.png")
            
        except Exception as e:
            print(f"Failed {opt}: {e}")
    
    print_table(results, "ViT-S/16 Final")
    plot_results(results, "ViT-S/16 on ImageNet", RESULTS_DIR / "vit_s16_final.png")
    
    with open(RESULTS_DIR / "vit_s16_all.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    return results

# Run
# vit_s_results = run_vit_s16_experiments()

In [ ]:
# =============================================================================
# CELL 10: ViT-B/16 Experiments
# =============================================================================

def run_vit_b16_experiments():
    print("\n" + "#"*70)
    print("# ViT-B/16 on ImageNet (300 epochs)")
    print("#"*70)
    
    results = {}
    
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        try:
            result = train_model(
                model_name="vit_b16", opt_name=opt,
                epochs=300, batch_size=1024,
                lr=1e-3, wd=0.05, warmup_epochs=30,
                use_randaugment=True, use_mixup=True, label_smoothing=0.1,
            )
            results[f"vit_b16_{opt}"] = result
            
            print_table(results, "ViT-B/16 Progress")
            plot_results(results, "ViT-B/16 Progress", RESULTS_DIR / "vit_b16_progress.png")
            
        except Exception as e:
            print(f"Failed {opt}: {e}")
    
    print_table(results, "ViT-B/16 Final")
    plot_results(results, "ViT-B/16 on ImageNet", RESULTS_DIR / "vit_b16_final.png")
    
    with open(RESULTS_DIR / "vit_b16_all.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    return results

# Run
# vit_b_results = run_vit_b16_experiments()

In [ ]:
# =============================================================================
# CELL 11: Run All
# =============================================================================

def run_all():
    all_results = {}
    all_results['resnet50'] = run_resnet50_experiments()
    all_results['vit_s16'] = run_vit_s16_experiments()
    all_results['vit_b16'] = run_vit_b16_experiments()
    
    with open(RESULTS_DIR / "all_results.json", 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print("\n" + "="*70)
    print("ALL EXPERIMENTS COMPLETE!")
    print("="*70)
    return all_results

# all_results = run_all()

In [ ]:
print("""
================================================================================
RLO EXPERIMENTS - READY
================================================================================

Fixes applied:
  ✓ torch.compile DISABLED (incompatible with DataParallel)
  ✓ num_workers=4 (reduced from 64)
  ✓ Memory optimizations

Still fast due to:
  - TF32 + bfloat16 mixed precision
  - Large batch sizes (4096 for ResNet, 2048 for ViT)
  - 8x B200 parallelism

To run:
  1. quick_result = quick_test()           # Verify everything works
  2. resnet_results = run_resnet50_experiments()   # ~15 hours
  3. vit_s_results = run_vit_s16_experiments()     # ~40 hours
  4. vit_b_results = run_vit_b16_experiments()     # ~60 hours

Or run all: all_results = run_all()

================================================================================
""")